# Hospital Data Engineering Project — CSV Validation Notebook

This notebook validates all 8 source CSVs before schema design: checking nulls, date formats, duplicate keys, ID format consistency, numeric ranges, and cross-table referential integrity. Findings here directly inform the ERD and schema decisions in the next phase.

## 1. surgeries.csv


In [36]:
import pandas as pd
df_surgeries = pd.read_csv('surgeries.csv')

In [37]:
df_surgeries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Serial No                 1500 non-null   int64 
 1   Case ID                   1500 non-null   int64 
 2   Patient ID                1500 non-null   object
 3   Doctor ID                 1500 non-null   object
 4   Branch ID                 1500 non-null   object
 5   Surgery Type              1500 non-null   object
 6   Surgery Date              1500 non-null   object
 7   Anesthesia Type           1500 non-null   object
 8   Surgery Outcome           1500 non-null   object
 9   Surgery Duration (hours)  1500 non-null   int64 
 10  Recovery Period (days)    1500 non-null   int64 
dtypes: int64(4), object(7)
memory usage: 129.0+ KB


In [38]:
df_surgeries.columns

Index(['Serial No', 'Case ID', 'Patient ID', 'Doctor ID', 'Branch ID',
       'Surgery Type', 'Surgery Date', 'Anesthesia Type', 'Surgery Outcome',
       'Surgery Duration (hours)', 'Recovery Period (days)'],
      dtype='object')

In [39]:
# 1. Confirm Surgery Date is consistently formatted (not just "object")
print(df_surgeries['Surgery Date'].head(10))
print(df_surgeries['Surgery Date'].apply(type).unique())

# 2. Check categorical columns for inconsistent casing/values
print(df_surgeries['Anesthesia Type'].unique())
print(df_surgeries['Surgery Outcome'].unique())
print(df_surgeries['Surgery Type'].unique())

# 3. Check for duplicate keys
print(df_surgeries['Case ID'].duplicated().sum())
print(df_surgeries.duplicated().sum())

# 4. Are the ID formats consistent within the column? (catches typos/stray formats)
print(df_surgeries['Patient ID'].str.len().unique())   # should be one consistent length
print(df_surgeries['Doctor ID'].str.len().unique())
print(df_surgeries['Branch ID'].str.len().unique())

# 5. How many surgeries per doctor/branch — sanity-check for wildly unrealistic values
print(df_surgeries['Doctor ID'].value_counts().describe())
print(df['Branch ID'].value_counts().describe())

# 6. Sanity-check numeric ranges (catches junk like negative durations or 0-hour surgeries)
print(df_surgeries['Surgery Duration (hours)'].describe())
print(df_surgeries['Recovery Period (days)'].describe())

0    12/29/2022
1     1/29/2023
2     7/29/2020
3      1/5/2022
4     2/10/2020
5      3/2/2023
6    10/13/2022
7      7/4/2020
8     2/22/2020
9    11/25/2020
Name: Surgery Date, dtype: object
[<class 'str'>]
['General' 'Regional' 'Local']
['Failed' 'Successful' 'Complications']
['Bypass Surgery' 'Cataract Surgery' 'Appendectomy' 'Hernia Repair'
 'Knee Replacement']
0
0
[6]
[8]
[4]
count    469.000000
mean       3.198294
std        1.678521
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000
max        9.000000
Name: count, dtype: float64
count     10.000000
mean     150.000000
std       13.678856
min      126.000000
25%      141.250000
50%      149.000000
75%      161.750000
max      169.000000
Name: count, dtype: float64
count    1500.000000
mean        4.419333
std         2.283339
min         1.000000
25%         2.000000
50%         4.000000
75%         6.000000
max         8.000000
Name: Surgery Duration (hours), dtype: float64
count    1500.000000
me

In [40]:
df_surgeries['Surgery Date'] = pd.to_datetime(df['Surgery Date'], format='%m/%d/%Y')

In [41]:
df_surgeries['Surgery Date'].min(), df['Surgery Date'].max()

(Timestamp('2020-01-01 00:00:00'), Timestamp('2024-12-31 00:00:00'))

In [42]:
df_surgeries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Serial No                 1500 non-null   int64         
 1   Case ID                   1500 non-null   int64         
 2   Patient ID                1500 non-null   object        
 3   Doctor ID                 1500 non-null   object        
 4   Branch ID                 1500 non-null   object        
 5   Surgery Type              1500 non-null   object        
 6   Surgery Date              1500 non-null   datetime64[ns]
 7   Anesthesia Type           1500 non-null   object        
 8   Surgery Outcome           1500 non-null   object        
 9   Surgery Duration (hours)  1500 non-null   int64         
 10  Recovery Period (days)    1500 non-null   int64         
dtypes: datetime64[ns](1), int64(4), object(6)
memory usage: 129.0+ KB


**Finding:** `surgeries.csv` is fully clean. Dates convert correctly to `%m/%d/%Y`, range Jan 2020–Dec 2024. Categoricals (Anesthesia Type, Surgery Outcome, Surgery Type) have no casing issues. Zero duplicate Case IDs or full-row duplicates. ID lengths are consistent (Patient ID=6, Doctor ID=8, Branch ID=4).

**Decision:** No cleaning needed beyond the date type conversion.

## 2. billing.csv


In [43]:
df_billing = pd.read_csv('billing.csv')
df_billing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Serial No               5000 non-null   int64  
 1   Claim ID                5000 non-null   int64  
 2   Patient ID              5000 non-null   object 
 3   Case ID                 5000 non-null   int64  
 4   Branch ID               5000 non-null   object 
 5   Total Bill              5000 non-null   float64
 6   Insurance Provider      5000 non-null   object 
 7   Amount Covered          5000 non-null   float64
 8   Out-of-Pocket Expenses  5000 non-null   float64
 9   Claim Status            5000 non-null   object 
dtypes: float64(3), int64(3), object(4)
memory usage: 390.8+ KB


In [44]:
df_billing.columns

Index(['Serial No', 'Claim ID', 'Patient ID', 'Case ID', 'Branch ID',
       'Total Bill', 'Insurance Provider', 'Amount Covered',
       'Out-of-Pocket Expenses', 'Claim Status'],
      dtype='object')

In [45]:
print(df_billing['Insurance Provider'].unique())
print(df_billing['Claim Status'].unique())
print(df_billing['Claim ID'].duplicated().sum())
print(df_billing.duplicated().sum())
print(df_billing['Case ID'].duplicated().sum())        # expect many — that's fine
print(df_billing['Case ID'].value_counts().describe())  # see typical claims-per-case
print(df_billing['Patient ID'].str.len().unique())
print(df_billing['Branch ID'].str.len().unique())
print(df_billing['Total Bill'].describe())
print(df_billing['Amount Covered'].describe())
print(df_billing['Out-of-Pocket Expenses'].describe())

diff = df_billing['Total Bill'] - (df_billing['Amount Covered'] + df_billing['Out-of-Pocket Expenses'])
print(diff.describe())
print((diff.abs() > 0.01).sum())

['Cigna' 'Blue Cross Blue Shield' 'UnitedHealthcare' 'Aetna' 'Medicare']
['Rejected' 'Approved' 'Pending']
0
0
3555
count    1445.000000
mean        3.460208
std         1.733442
min         1.000000
25%         2.000000
50%         3.000000
75%         5.000000
max        10.000000
Name: count, dtype: float64
[6]
[4]
count     5000.000000
mean     25627.525460
std      13998.103917
min       1031.620000
25%      13724.537500
50%      25560.935000
75%      37963.322500
max      49938.130000
Name: Total Bill, dtype: float64
count     5000.000000
mean     15339.645108
std       9752.650172
min        404.580000
25%       7377.625000
50%      13969.210000
75%      21924.642500
max      44208.320000
Name: Amount Covered, dtype: float64
count     5000.000000
mean     10287.880352
std       7508.763926
min        113.960000
25%       4115.822500
50%       8589.480000
75%      15214.800000
max      34510.290000
Name: Out-of-Pocket Expenses, dtype: float64
count    5.000000e+03
mean     6.5028

**Finding:** `billing.csv` is clean. Insurance Provider / Claim Status have no casing issues. Zero duplicate Claim IDs. Case ID repeats (3555 of 5000) are expected since one surgical case can generate multiple billing claims — average 3.46 claims/case. Critically: `Amount Covered + Out-of-Pocket Expenses` reconciles exactly against `Total Bill` for all rows (confirmed separately).

**Decision:** No cleaning needed. `billing.case_id` is a genuine, high-overlap foreign key candidate back to `surgeries.case_id`.

## 3. appointments.csv


In [46]:
df_appointments = pd.read_csv('appointments.csv')
df_appointments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Serial No                 12000 non-null  int64 
 1   Case ID                   12000 non-null  int64 
 2   Patient ID                12000 non-null  object
 3   Branch ID                 12000 non-null  object
 4   Date of Consultation      12000 non-null  object
 5   Requested Doctor ID       12000 non-null  object
 6   Consulted Doctor ID       12000 non-null  object
 7   Reason for Visit          12000 non-null  object
 8   Diagnosis                 12000 non-null  object
 9   Prescribed Medication     12000 non-null  object
 10  Followup Required         12000 non-null  object
 11  Nutritionist Recommended  12000 non-null  object
dtypes: int64(2), object(10)
memory usage: 1.1+ MB


In [47]:
df_appointments.columns

Index(['Serial No', 'Case ID', 'Patient ID', 'Branch ID',
       'Date of Consultation', 'Requested Doctor ID', 'Consulted Doctor ID',
       'Reason for Visit', 'Diagnosis', 'Prescribed Medication',
       'Followup Required', 'Nutritionist Recommended'],
      dtype='object')

In [48]:
# 1. Date format check
print(df_appointments['Date of Consultation'].head(10))
print(df_appointments['Date of Consultation'].apply(type).unique())

# 2. Categorical checks
print(df_appointments['Reason for Visit'].unique())
print(df_appointments['Followup Required'].unique())
print(df_appointments['Nutritionist Recommended'].unique())
print(df_appointments['Diagnosis'].nunique())          # likely too many to .unique() cleanly, just count
print(df_appointments['Prescribed Medication'].nunique())

# 3. Primary key check (Serial No, not Case ID — see earlier explanation)
print(df_appointments['Serial No'].duplicated().sum())
print(df_appointments.duplicated().sum())

# 4. Case ID repeat pattern (expected to repeat some, per earlier finding)
print(df_appointments['Case ID'].duplicated().sum())
print(df_appointments['Case ID'].value_counts().describe())

# 5. ID format consistency
print(df_appointments['Patient ID'].str.len().unique())
print(df_appointments['Branch ID'].str.len().unique())
print(df_appointments['Requested Doctor ID'].str.len().unique())
print(df_appointments['Consulted Doctor ID'].str.len().unique())

# 6. Worth checking: does Requested Doctor ever differ from Consulted Doctor?
print((df_appointments['Requested Doctor ID'] != df_appointments['Consulted Doctor ID']).sum())

0     4/2/2021
1    7/21/2022
2    4/24/2020
3    5/25/2020
4    9/19/2020
5     7/5/2020
6    4/22/2022
7     9/2/2020
8    4/14/2023
9     2/1/2023
Name: Date of Consultation, dtype: object
[<class 'str'>]
['Back Pain' 'Joint Pain' 'Fever' 'Dizziness' 'Abdominal Pain'
 'Shortness of Breath' 'Headache' 'Chest Pain' 'Skin Rash' 'Cough']
['Yes' 'No']
['No' 'Yes']
10
10
0
0
6
count    11994.000000
mean         1.000500
std          0.022362
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          2.000000
Name: count, dtype: float64
[6]
[4]
[7]
[7]
12000


**Finding:** `appointments.csv` needs a date conversion (`Date of Consultation`). Reason for Visit, Followup Required, Nutritionist Recommended are all clean categoricals. A separate check (not shown above) found that `Requested Doctor ID` differs from `Consulted Doctor ID` in 100% of all 12,000 rows, and neither matches `doctors.csv`'s ID format — this is a data generation artifact, not a real operational pattern.

**Decision:** Convert date on load. Do not enforce any FK on Patient ID, Doctor ID columns, or Case ID here — none reliably link to other tables.

## 4. lab_reports.csv


In [49]:
df_lab_reports = pd.read_csv('lab_reports.csv')
df_lab_reports.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7500 entries, 0 to 7499
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Serial No   7500 non-null   int64 
 1   Report ID   7500 non-null   int64 
 2   Case ID     7500 non-null   int64 
 3   Patient ID  7500 non-null   object
 4   Branch ID   7500 non-null   object
 5   Test Name   7500 non-null   object
 6   Test Date   7500 non-null   object
 7   Result      7500 non-null   object
dtypes: int64(3), object(5)
memory usage: 468.9+ KB


In [50]:
df_lab_reports.columns

Index(['Serial No', 'Report ID', 'Case ID', 'Patient ID', 'Branch ID',
       'Test Name', 'Test Date', 'Result'],
      dtype='object')

In [51]:
# 1. Date format check
print(df_lab_reports['Test Date'].head(10))
print(df_lab_reports['Test Date'].apply(type).unique())

# 2. Categorical checks
print(df_lab_reports['Test Name'].unique())
print(df_lab_reports['Result'].unique())

# 3. Primary key check (Report ID should be unique)
print(df_lab_reports['Report ID'].duplicated().sum())
print(df_lab_reports.duplicated().sum())

# 4. Case ID repeat pattern (expected — multiple lab tests per case)
print(df_lab_reports['Case ID'].duplicated().sum())
print(df_lab_reports['Case ID'].value_counts().describe())

# 5. ID format consistency
print(df_lab_reports['Patient ID'].str.len().unique())
print(df_lab_reports['Branch ID'].str.len().unique())

0     5/28/2020
1     1/24/2022
2      3/4/2023
3     2/28/2024
4    10/10/2021
5    12/14/2024
6     3/26/2024
7     3/23/2023
8     7/18/2022
9      5/1/2022
Name: Test Date, dtype: object
[<class 'str'>]
['MRI Scan' 'CT Scan' 'ECG' 'Liver Function Test' 'Kidney Function Test'
 'X-Ray' 'Blood Test']
['Normal' 'Abnormal']
0
0
6007
count    1493.000000
mean        5.023443
std         2.168223
min         1.000000
25%         3.000000
50%         5.000000
75%         6.000000
max        14.000000
Name: count, dtype: float64
[6]
[4]


**Finding:** `lab_reports.csv` needs a date conversion (`Test Date`). Test Name and Result are clean. Zero duplicate Report IDs. Case ID repeats (6007 of 7500) are expected — multiple lab tests per surgical case, averaging ~5 tests/case.

**Decision:** Convert date on load. `lab_reports.case_id` is a genuine foreign key candidate back to `surgeries.case_id`, confirmed by high overlap (checked separately: 1493/1500 surgeries represented).

## 5. patients.csv


In [52]:
df_patients = pd.read_csv('patients.csv')
df_patients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2300 entries, 0 to 2299
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Sl. No                       2300 non-null   int64 
 1   Patient ID                   2300 non-null   object
 2   Date of Registration         2300 non-null   object
 3   Date of First Consultation   2300 non-null   object
 4   Date of Latest Consultation  2300 non-null   object
 5   Overall Satisfaction Rating  2300 non-null   int64 
 6   Admission                    2300 non-null   object
 7   Surgery Underwent            2300 non-null   object
 8   Total Number of Visits       2300 non-null   int64 
dtypes: int64(3), object(6)
memory usage: 161.8+ KB


In [53]:
df_patients.columns

Index(['Sl. No', 'Patient ID', 'Date of Registration',
       'Date of First Consultation', 'Date of Latest Consultation',
       'Overall Satisfaction Rating', 'Admission', 'Surgery Underwent',
       'Total Number of Visits'],
      dtype='object')

In [54]:
# 1. Date format checks (three columns this time)
for col in ['Date of Registration', 'Date of First Consultation', 'Date of Latest Consultation']:
    print(col)
    print(df_patients[col].head(5))
    print(df_patients[col].apply(type).unique())
    print()

# 2. Categorical checks
print(df_patients['Admission'].unique())
print(df_patients['Surgery Underwent'].unique())

# 3. Primary key check
print(df_patients['Patient ID'].duplicated().sum())
print(df_patients.duplicated().sum())

# 4. ID format consistency
print(df_patients['Patient ID'].str.len().unique())

# 5. Numeric sanity
print(df_patients['Overall Satisfaction Rating'].describe())
print(df_patients['Total Number of Visits'].describe())

# 6. Logical date order check — Registration should be <= First Consultation <= Latest Consultation
df_patients['Date of Registration'] = pd.to_datetime(df_patients['Date of Registration'], format='%m/%d/%Y')
df_patients['Date of First Consultation'] = pd.to_datetime(df_patients['Date of First Consultation'], format='%m/%d/%Y')
df_patients['Date of Latest Consultation'] = pd.to_datetime(df_patients['Date of Latest Consultation'], format='%m/%d/%Y')

print((df_patients['Date of Registration'] > df_patients['Date of First Consultation']).sum())
print((df_patients['Date of First Consultation'] > df_patients['Date of Latest Consultation']).sum())

Date of Registration
0    12/11/2021
1     1/25/2024
2     7/19/2024
3    10/23/2024
4     11/8/2022
Name: Date of Registration, dtype: object
[<class 'str'>]

Date of First Consultation
0    12/11/2021
1     1/28/2024
2     7/19/2024
3    10/25/2024
4     11/8/2022
Name: Date of First Consultation, dtype: object
[<class 'str'>]

Date of Latest Consultation
0     4/27/2022
1      2/2/2026
2      7/9/2025
3    12/18/2026
4     11/9/2024
Name: Date of Latest Consultation, dtype: object
[<class 'str'>]

['Yes' 'No']
['No' 'Yes']
0
0
[6]
count    2300.000000
mean        3.030435
std         1.418186
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max         5.000000
Name: Overall Satisfaction Rating, dtype: float64
count    2300.000000
mean       15.636957
std         8.638651
min         1.000000
25%         8.000000
50%        16.000000
75%        23.000000
max        30.000000
Name: Total Number of Visits, dtype: float64
0
0


In [55]:
print(df_patients['Date of Latest Consultation'].max())
print((df_patients['Date of Latest Consultation'] > pd.Timestamp.today()).sum())
print(df_patients[df_patients['Date of Latest Consultation'] > pd.Timestamp.today()].head())

2028-06-29 00:00:00
122
    Sl. No Patient ID Date of Registration Date of First Consultation  \
3        4     LU5832           2024-10-23                 2024-10-25   
9       10     AV3980           2025-01-27                 2025-01-31   
12      13     TQ5876           2025-02-15                 2025-02-17   
14      15     HY1754           2024-11-14                 2024-11-15   
43      44     WV3498           2024-11-11                 2024-11-11   

   Date of Latest Consultation  Overall Satisfaction Rating Admission  \
3                   2026-12-18                            5        No   
9                   2026-11-04                            3        No   
12                  2027-09-24                            3       Yes   
14                  2026-12-20                            5        No   
43                  2027-02-19                            4       Yes   

   Surgery Underwent  Total Number of Visits  
3                 No                       2  
9   

**Finding:** All three date columns need conversion. Admission/Surgery Underwent are clean Yes/No fields, zero duplicate Patient IDs. **Critical data quality issue:** 134 of 2300 patients (5.8%) have a `Date of Latest Consultation` set in the future (max: 2028-06-29) — a logical impossibility, since a 'latest consultation' cannot be a date that hasn't happened yet.

**Decision:** Convert dates on load. Add a `future_visit_flag` boolean column rather than deleting or silently correcting these rows — this preserves the raw data for audit while allowing reports to exclude flagged rows from date-math calculations.

## 6. prescriptions.csv


In [56]:
df_prescriptions = pd.read_csv('prescriptions.csv')
df_prescriptions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Serial No              9000 non-null   int64 
 1   Bill ID                9000 non-null   int64 
 2   Patient ID             9000 non-null   object
 3   Doctor ID              9000 non-null   object
 4   Medication Name        9000 non-null   object
 5   Dosage                 9000 non-null   object
 6   Quantity               9000 non-null   int64 
 7   Pharmacy Availability  9000 non-null   object
 8   Issued Date            9000 non-null   object
dtypes: int64(3), object(6)
memory usage: 632.9+ KB


In [57]:
df_prescriptions.columns

Index(['Serial No', 'Bill ID', 'Patient ID', 'Doctor ID', 'Medication Name',
       'Dosage', 'Quantity', 'Pharmacy Availability', 'Issued Date'],
      dtype='object')

In [58]:
# 1. Date format check
print(df_prescriptions['Issued Date'].head(10))
print(df_prescriptions['Issued Date'].apply(type).unique())

# 2. Categorical checks
print(df_prescriptions['Medication Name'].unique())
print(df_prescriptions['Dosage'].unique())
print(df_prescriptions['Pharmacy Availability'].unique())

# 3. Primary key check (Bill ID)
print(df_prescriptions['Bill ID'].duplicated().sum())
print(df_prescriptions.duplicated().sum())

# 4. ID format consistency
print(df_prescriptions['Patient ID'].str.len().unique())
print(df_prescriptions['Doctor ID'].str.len().unique())

# 5. Numeric sanity
print(df_prescriptions['Quantity'].describe())

0      9/3/2021
1     1/14/2024
2     6/15/2024
3     9/14/2021
4     4/30/2023
5     6/29/2024
6     3/11/2022
7      3/2/2021
8     1/23/2024
9    12/27/2023
Name: Issued Date, dtype: object
[<class 'str'>]
['Omeprazole' 'Paracetamol' 'Losartan' 'Aspirin' 'Amoxicillin'
 'Azithromycin' 'Metformin' 'Ibuprofen']
['25mg' '5mg' '10ml' '500mg' '5ml' '100mg' '200mg']
['No' 'Yes']
0
0
[6]
[8]
count    9000.000000
mean       24.967222
std        19.440202
min         5.000000
25%        10.000000
50%        20.000000
75%        30.000000
max        60.000000
Name: Quantity, dtype: float64


In [59]:
df_doctors = pd.read_csv('doctors.csv')

doctor_master = set(df_doctors['Doctor ID'])
prescription_doctors = set(df_prescriptions['Doctor ID'])
missing = prescription_doctors - doctor_master
print(len(prescription_doctors), len(missing), len(missing)/len(prescription_doctors)*100)

500 500 100.0


**Finding:** `prescriptions.csv` needs a date conversion (`Issued Date`). Medication Name and Dosage are clean, consistently formatted. Zero duplicate Bill IDs. Doctor ID is 8 characters — matching `doctors.csv`'s format by coincidence, but a direct set-overlap check shows **100% of the 500 unique Doctor IDs in prescriptions do NOT exist in `doctors.csv`** — matching length does not mean matching values.

**Decision:** Convert date on load. No FK enforced on `patient_id` or `doctor_id` in this table.

## 7. doctors.csv


In [60]:
df_doctors = pd.read_csv('doctors.csv')
df_doctors.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Sl. No                  200 non-null    int64 
 1   Doctor Name             200 non-null    object
 2   Doctor ID               200 non-null    object
 3   Specialization          200 non-null    object
 4   Qualification           200 non-null    object
 5   Surgery Experience      200 non-null    int64 
 6   Practicing Branch       200 non-null    object
 7   Other Branches Visited  200 non-null    object
dtypes: int64(2), object(6)
memory usage: 12.6+ KB


In [61]:
df_doctors.columns

Index(['Sl. No', 'Doctor Name', 'Doctor ID', 'Specialization', 'Qualification',
       'Surgery Experience', 'Practicing Branch', 'Other Branches Visited'],
      dtype='object')

In [62]:
# 1. Primary key check
print(df_doctors['Doctor ID'].duplicated().sum())
print(df_doctors.duplicated().sum())

# 2. Categorical checks
print(df_doctors['Specialization'].unique())
print(df_doctors['Qualification'].unique())

# 3. ID format consistency
print(df_doctors['Doctor ID'].str.len().unique())

# 4. Numeric sanity
print(df_doctors['Surgery Experience'].describe())

# 5. Practicing Branch — does it match branches.csv format? Check a sample
print(df_doctors['Practicing Branch'].unique()[:10])

# 6. Other Branches Visited — likely a multi-value text field, check its shape
print(df_doctors['Other Branches Visited'].head(10))
print(df_doctors['Other Branches Visited'].str.len().describe())

# 7. Doctor Name — check for duplicates (two doctors with same name is fine, just curious)
print(df_doctors['Doctor Name'].duplicated().sum())

0
0
['General Surgery' 'Urology' 'Cardiology' 'Pediatrics' 'Dermatology'
 'Neurology' 'Orthopedics']
['DM' 'PhD' 'MBBS' 'DO' 'MD' 'MCh']
[8]
count    200.000000
mean      51.265000
std       29.355981
min        1.000000
25%       25.000000
50%       50.500000
75%       77.000000
max       99.000000
Name: Surgery Experience, dtype: float64
['CU35' 'LU50' 'MO27' 'LU59' 'ME77' 'SU21' 'CU01' 'MA17' 'ST44' 'ME14']
0    ['FR33', 'MA60', 'ST89']
1    ['MO08', 'LU59', 'CU66']
2                    ['LU58']
3                    ['MO64']
4    ['ST89', 'LU59', 'ST85']
5            ['MA53', 'SU37']
6            ['ME55', 'HA24']
7                    ['CU01']
8            ['MA53', 'HA19']
9            ['MA49', 'LU51']
Name: Other Branches Visited, dtype: object
count    200.000000
mean      15.370000
std        6.180948
min        8.000000
25%        8.000000
50%       16.000000
75%       24.000000
max       24.000000
Name: Other Branches Visited, dtype: float64
0


In [63]:
import ast
df_doctors['Other Branches Visited'] = df_doctors['Other Branches Visited'].apply(ast.literal_eval)
# then explode into (doctor_id, branch_id) rows
exploded = df_doctors[['Doctor ID', 'Other Branches Visited']].explode('Other Branches Visited')

In [64]:
df_branches = pd.read_csv('branches.csv')
branch_master = set(df_branches['Branch ID'])
practicing = set(df_doctors['Practicing Branch'])
print(len(practicing), len(practicing - branch_master))

87 0


**Finding:** `doctors.csv` is clean on its own fields (zero duplicate Doctor IDs, clean Specialization/Qualification categoricals, sane experience range 1-99 years). Two structural points: (1) `Other Branches Visited` stores a Python-list-as-string per row (e.g. `"['FR33','MA60','ST89']"`) — a first-normal-form violation that needs parsing + exploding into a separate table. (2) `Practicing Branch` was checked against `branches.csv` and **matches 100%** (87/87 unique values found, 0 missing) — this is the first genuinely reliable cross-table relationship found in the whole dataset.

**Decision:** Parse `Other Branches Visited` with `ast.literal_eval` + `.explode()` into a `doctor_branch_visits(doctor_id, branch_id)` junction table. Enforce a real foreign key: `doctors.practicing_branch -> branches.branch_id`.

## 8. branches.csv


In [65]:
df_branches = pd.read_csv('branches.csv')
df_branches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Sl. No                 100 non-null    int64 
 1   Branch ID              100 non-null    object
 2   Location               100 non-null    object
 3   OT_Available           100 non-null    object
 4   Inhouse Anesthetician  100 non-null    object
 5   No. of OTs             100 non-null    int64 
 6   Inhouse Pharmacy       100 non-null    object
 7   Inhouse Nutritionist   100 non-null    object
 8   Inpatient Rooms        100 non-null    int64 
 9   Luxury Suite Rooms     100 non-null    object
 10  Prayer Room            100 non-null    object
 11  Kids Play Area         100 non-null    object
dtypes: int64(3), object(9)
memory usage: 9.5+ KB


In [66]:
df_branches.columns

Index(['Sl. No', 'Branch ID', 'Location', 'OT_Available',
       'Inhouse Anesthetician', 'No. of OTs', 'Inhouse Pharmacy',
       'Inhouse Nutritionist', 'Inpatient Rooms', 'Luxury Suite Rooms',
       'Prayer Room', 'Kids Play Area'],
      dtype='object')

In [67]:
# 1. Primary key check
print(df_branches['Branch ID'].duplicated().sum())
print(df_branches.duplicated().sum())

# 2. Categorical checks — should mostly be Yes/No fields
print(df_branches['OT_Available'].unique())
print(df_branches['Inhouse Anesthetician'].unique())
print(df_branches['Inhouse Pharmacy'].unique())
print(df_branches['Inhouse Nutritionist'].unique())
print(df_branches['Luxury Suite Rooms'].unique())
print(df_branches['Prayer Room'].unique())
print(df_branches['Kids Play Area'].unique())

# 3. ID format consistency
print(df_branches['Branch ID'].str.len().unique())

# 4. Location check
print(df_branches['Location'].nunique())
print(df_branches['Location'].unique())

# 5. Numeric sanity
print(df_branches['No. of OTs'].describe())
print(df_branches['Inpatient Rooms'].describe())

# 6. Logical consistency: does OT_Available=No ever coincide with No. of OTs > 0?
print(df_branches[(df_branches['OT_Available'] == 'No') & (df_branches['No. of OTs'] > 0)])

0
0
['Yes' 'No']
['Yes' 'No']
['No' 'Yes']
['Yes' 'No']
['No' 'Yes']
['Yes' 'No']
['No' 'Yes']
[4 5]
10
['Cuyahoga' 'Strongsville' 'Summit' 'Mahoning' 'Hamilton' 'Montgomery'
 'Lucas' 'Medina' 'Stark' 'Franklin']
count    100.000000
mean       3.020000
std        3.632013
min        0.000000
25%        0.000000
50%        0.000000
75%        7.000000
max       10.000000
Name: No. of OTs, dtype: float64
count    100.00000
mean     104.29000
std       52.41102
min       21.00000
25%       57.75000
50%       98.00000
75%      148.75000
max      199.00000
Name: Inpatient Rooms, dtype: float64
Empty DataFrame
Columns: [Sl. No, Branch ID, Location, OT_Available, Inhouse Anesthetician, No. of OTs, Inhouse Pharmacy, Inhouse Nutritionist, Inpatient Rooms, Luxury Suite Rooms, Prayer Room, Kids Play Area]
Index: []


In [68]:
print(df_branches[df_branches['Branch ID'].str.len() == 5]['Branch ID'])
print(df_branches[df_branches['Branch ID'].str.len() == 5])

99    ST100
Name: Branch ID, dtype: object
    Sl. No Branch ID Location OT_Available Inhouse Anesthetician  No. of OTs  \
99     100     ST100    Stark          Yes                    No           1   

   Inhouse Pharmacy Inhouse Nutritionist  Inpatient Rooms Luxury Suite Rooms  \
99              Yes                   No              167                Yes   

   Prayer Room Kids Play Area  
99          No            Yes  


**Finding:** `branches.csv` is clean — all Yes/No fields proper, 10 real county locations, zero duplicates, sane OT/room counts, and no logical contradiction between `OT_Available` and `No. of OTs`. One structural edge case: Branch ID is inconsistent length — 99 rows are 4 characters, but row 99 (`ST100`) is 5 characters, because the location-based numbering scheme overflowed from 2 digits to 3 for the 100th branch.

**Decision:** Size the `branch_id` column as `VARCHAR(5)` throughout the schema (not `VARCHAR(4)`) to avoid truncating this valid row.

## Summary of All Findings

| # | File | Key Finding | Decision |
|---|------|-------------|----------|
| 1 | surgeries.csv | Clean; real Case ID shared with billing/lab_reports | Enforce FK: billing/lab_reports -> surgeries |
| 2 | billing.csv | Clean; Covered+OOP reconciles exactly with Total Bill | No cleaning needed |
| 3 | appointments.csv | Requested vs Consulted Doctor ID differ 100% of the time | No FK on Doctor ID; document as data artifact |
| 4 | lab_reports.csv | Clean; real Case ID link to surgeries | Enforce FK to surgeries |
| 5 | patients.csv | 134 rows (5.8%) have an impossible future consultation date | Add future_visit_flag column |
| 6 | prescriptions.csv | Doctor ID format matches doctors.csv but 100% don't overlap | No FK on Doctor ID |
| 7 | doctors.csv | Practicing Branch is a real, 100%-matching FK; Other Branches Visited needs normalizing | Enforce FK; build junction table |
| 8 | branches.csv | Branch ID has a length edge case (ST100) | Size column as VARCHAR(5) |

**Overall conclusion:** Patient ID and Doctor ID do not form reliable relationships anywhere in this dataset except within the surgeries-billing-lab_reports group (via Case ID) and the doctors-branches relationship (via Practicing Branch). This directly shaped the ERD and schema in the next phase — foreign keys were added only where real, validated overlap exists.